# 00 - Encoder smoke test

Loads MERT-v1-95M, embeds 5 audio files, prints the cosine similarity matrix.
First run will download ~400 MB of model weights to `models/hf_cache/`.

**Pre-req**: a directory with at least 5 audio files (FLAC/MP3/Opus/...).
Set `AUDIO_DIR` below.

In [ ]:
from pathlib import Path
import sys

# Allow `import encoder...` when the notebook is launched from the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

AUDIO_DIR = ROOT / "data" / "raw" / "fma_small"  # adjust to taste
CACHE_DIR = ROOT / "models" / "hf_cache"
print("audio:", AUDIO_DIR, "exists:", AUDIO_DIR.exists())
print("cache:", CACHE_DIR)

In [ ]:
from encoder.batching import iterate_files
from encoder.mert_encoder import MertEncoder

files = list(iterate_files(AUDIO_DIR))[:5]
for f in files:
    print(f)
assert files, f"no audio files under {AUDIO_DIR}"

In [ ]:
import numpy as np

encoder = MertEncoder(cache_dir=CACHE_DIR, device="cpu")
embs = np.stack([encoder.embed_file(f) for f in files], axis=0)
print(embs.shape, embs.dtype)
print("L2 norms (should be ~1.0):", np.linalg.norm(embs, axis=1))

In [ ]:
import pandas as pd

sim = embs @ embs.T
labels = [f.name for f in files]
pd.DataFrame(sim, index=labels, columns=labels).round(3)

Diagonal should be ~1.0. Off-diagonal: similar songs (same album, same genre) should be noticeably higher than different-genre pairs.